In [6]:
import numpy as np
import pandas as pd

In [35]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [36]:
df=pd.read_csv('covid_toy.csv')

In [37]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [38]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [39]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [40]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(df.drop(columns=['has_covid']),
                                              df['has_covid'],
                                              test_size=0.2)

In [41]:
X_train

,age,gender,fever,cough,city
3,31,Female,98.0,Mild,Kolkata
20,12,Male,98.0,Strong,Bangalore
7,20,Female,NaN,Strong,Mumbai
78,11,Male,100.0,Mild,Bangalore
56,71,Male,NaN,Strong,Kolkata
...,...,...,...,...,...
71,75,Female,104.0,Strong,Delhi
95,12,Female,104.0,Mild,Bangalore
13,64,Male,102.0,Mild,Bangalore
93,27,Male,100.0,Mild,Kolkata


## Without Transformer

In [42]:
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[['fever']])
X_test_fever=si.fit_transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [46]:

oe=OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough=oe.fit_transform(X_train[['cough']])
X_test_cough=oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [50]:
ohe=OneHotEncoder(drop='first')
X_train_gender_city=ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city=ohe.fit_transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 4)

In [52]:
X_train_age=X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age=X_test.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape

(80, 1)

In [54]:
print(type(X_train_gender_city))
print(X_train_gender_city)

<class 'scipy.sparse._csr.csr_matrix'>
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 91 stored elements and shape (80, 4)>
  Coords	Values
  (0, 2)	1.0
  (1, 0)	1.0
  (2, 3)	1.0
  (3, 0)	1.0
  (4, 0)	1.0
  (4, 2)	1.0
  (5, 0)	1.0
  (5, 2)	1.0
  (6, 0)	1.0
  (6, 1)	1.0
  (7, 2)	1.0
  (8, 2)	1.0
  (9, 2)	1.0
  (10, 2)	1.0
  (11, 0)	1.0
  (11, 2)	1.0
  (14, 0)	1.0
  (14, 3)	1.0
  (16, 3)	1.0
  (18, 0)	1.0
  (19, 0)	1.0
  (19, 2)	1.0
  (20, 2)	1.0
  (21, 1)	1.0
  (22, 0)	1.0
  :	:
  (58, 3)	1.0
  (59, 3)	1.0
  (61, 0)	1.0
  (61, 1)	1.0
  (62, 2)	1.0
  (63, 1)	1.0
  (64, 1)	1.0
  (65, 0)	1.0
  (65, 3)	1.0
  (66, 1)	1.0
  (68, 1)	1.0
  (69, 0)	1.0
  (69, 3)	1.0
  (70, 0)	1.0
  (71, 1)	1.0
  (72, 0)	1.0
  (72, 1)	1.0
  (73, 0)	1.0
  (73, 1)	1.0
  (74, 2)	1.0
  (75, 1)	1.0
  (77, 0)	1.0
  (78, 0)	1.0
  (78, 2)	1.0
  (79, 1)	1.0


In [56]:
X_train_gender_city = X_train_gender_city.toarray()
X_test_gender_city = X_test_gender_city.toarray()

AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [60]:
X_train_transformed=np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
X_test_transformed=np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)
X_train_transformed.shape

(80, 7)

## With Columns transform

In [65]:
from sklearn.compose import ColumnTransformer
transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first'),['gender','city'])
],remainder='passthrough')

In [67]:
transformer.fit_transform(X_train).shape

(80, 7)

In [68]:
transformer.fit_transform(X_test).shape

(20, 7)